<a name="title"></a>
# Live-Learning Research Agent with Perplexity (Search + Embeddings + Agent) and Qdrant

*Notebook by the [Perplexity](https://perplexity.ai) API team.*

In this cookbook we build a **single research agent that uses all three Perplexity APIs together** through the [`perplexity-haystack`](https://haystack.deepset.ai/integrations/perplexity) integration:

| Perplexity API | Haystack component | Role in this notebook |
|---|---|---|
| **Agent API** (`POST /v1/agent`) | `PerplexityChatGenerator` | The agent's reasoning model. OpenAI-Responses-compatible, so it plugs into Haystack's [`Agent`](https://docs.haystack.deepset.ai/docs/agent) without any glue. |
| **Search API** (`POST /search`) | `PerplexityWebSearch` | Ranked, cleaned, cited web results — exposed to the agent as a tool, replacing the SerperDev / Apify / DuckDuckGo + `LinkContentFetcher` chain other cookbooks build by hand. |
| **Embeddings API** (`POST /v1/embeddings`) | `PerplexityDocumentEmbedder` / `PerplexityTextEmbedder` | Indexes documents into a [Qdrant](https://haystack.deepset.ai/integrations/qdrant-document-store) store and embeds queries at retrieval time. |

The agent gets three tools — `retrieve_from_index`, `web_search`, `ingest_url` — and decides per-question whether to look at its existing index, search the live web, or grow the index with a new URL. The result is a knowledge base that *learns from the agent's own behaviour*, with citations attached to every web answer.


<a name="what-you-will-learn"></a>
## What you will build

1. A Qdrant-backed knowledge base seeded with a few Haystack documentation pages, embedded with `pplx-embed-v1-0.6b`.
2. A `web_search` tool wrapping `PerplexityWebSearch` so the agent can call the Perplexity Search API directly.
3. An `ingest_url` tool that takes a URL the agent picked from search results, fetches it, embeds it with `PerplexityDocumentEmbedder`, and writes it back to Qdrant.
4. A `retrieve_from_index` tool that embeds the query with `PerplexityTextEmbedder` and pulls the top-k from Qdrant.
5. A Haystack [`Agent`](https://docs.haystack.deepset.ai/docs/agent) driven by `PerplexityChatGenerator` that orchestrates the three tools.
6. A short evaluation showing the index growing across turns and answers carrying citations end-to-end.


<a name="setup"></a>
## 1. Setup

Install the integration packages. We use an in-memory Qdrant instance so the notebook runs anywhere (Colab included) without spinning up infrastructure.


In [ ]:
%pip install -q \
    "haystack-ai>=2.24.1" \
    "perplexity-haystack" \
    "qdrant-haystack" \
    "trafilatura"

<a name="api-keys"></a>
### 1.1 API key

You only need **one** API key for this notebook — your Perplexity key (covers the Agent API, Search API and Embeddings API). Get one at [perplexity.ai/account/api](https://www.perplexity.ai/account/api).


In [ ]:
import os
from getpass import getpass

if not os.environ.get("PERPLEXITY_API_KEY"):
    os.environ["PERPLEXITY_API_KEY"] = getpass("PERPLEXITY_API_KEY: ")

<a name="seed-kb"></a>
## 2. Seed the knowledge base with Perplexity Embeddings

We start with three short seed documents covering Haystack basics. They get embedded with `PerplexityDocumentEmbedder` (model `pplx-embed-v1-0.6b`, 1536-dim) and written into an in-memory Qdrant store.


In [ ]:
from haystack import Document
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore

document_store = QdrantDocumentStore(
    location=":memory:",
    index="research_agent",
    embedding_dim=1536,           # pplx-embed-v1-0.6b output dim
    similarity="cosine",
    return_embedding=False,
)

In [ ]:
seed_docs = [
    Document(
        content=(
            "Haystack is an open-source Python framework by deepset for building "
            "production LLM applications such as RAG pipelines and agentic workflows."
        ),
        meta={"source": "https://haystack.deepset.ai/", "title": "Haystack overview"},
    ),
    Document(
        content=(
            "A Haystack Agent is a tool-calling component that wraps a chat generator "
            "and a list of Tool objects. The agent loops: it asks the model what to do, "
            "executes the chosen tool, and feeds the result back until it produces a final answer."
        ),
        meta={"source": "https://docs.haystack.deepset.ai/docs/agent", "title": "Haystack Agent"},
    ),
    Document(
        content=(
            "The perplexity-haystack package ships PerplexityChatGenerator (Agent API), "
            "PerplexityTextEmbedder + PerplexityDocumentEmbedder (Embeddings API), and "
            "PerplexityWebSearch (Search API)."
        ),
        meta={
            "source": "https://haystack.deepset.ai/integrations/perplexity",
            "title": "perplexity-haystack integration",
        },
    ),
]

In [ ]:
from haystack import Pipeline
from haystack.components.writers import DocumentWriter
from haystack.document_stores.types import DuplicatePolicy
from haystack_integrations.components.embedders.perplexity import PerplexityDocumentEmbedder

indexing = Pipeline()
indexing.add_component("embedder", PerplexityDocumentEmbedder(model="pplx-embed-v1-0.6b"))
indexing.add_component(
    "writer",
    DocumentWriter(document_store=document_store, policy=DuplicatePolicy.OVERWRITE),
)
indexing.connect("embedder.documents", "writer.documents")

indexing.run({"embedder": {"documents": seed_docs}})
print(f"Indexed {document_store.count_documents()} seed documents.")

<a name="tools"></a>
## 3. Define the three tools

Each tool is a thin function wrapped with Haystack's [`Tool`](https://docs.haystack.deepset.ai/docs/tool) dataclass and a JSON-schema for arguments. We close over the document store and the embedder components so the tools stay pure functions from the agent's perspective.


<a name="tool-retrieve"></a>
### 3.1 `retrieve_from_index` — Perplexity Embeddings + Qdrant


In [ ]:
from haystack.tools import Tool
from haystack_integrations.components.embedders.perplexity import PerplexityTextEmbedder
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

_text_embedder = PerplexityTextEmbedder(model="pplx-embed-v1-0.6b")
_retriever = QdrantEmbeddingRetriever(document_store=document_store, top_k=4)


def retrieve_from_index(query: str, top_k: int = 4) -> dict:
    """Embed the query with Perplexity and pull the top-k matches from Qdrant."""
    query_emb = _text_embedder.run(text=query)["embedding"]
    hits = _retriever.run(query_embedding=query_emb, top_k=top_k)["documents"]
    return {
        "hits": [
            {
                "title": d.meta.get("title", ""),
                "source": d.meta.get("source", ""),
                "snippet": d.content[:500],
                "score": d.score,
            }
            for d in hits
        ]
    }


retrieve_tool = Tool(
    name="retrieve_from_index",
    description=(
        "Search the local Qdrant knowledge base (Perplexity embeddings). "
        "Use this FIRST for any question that might already be covered by indexed sources."
    ),
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Natural-language search query."},
            "top_k": {"type": "integer", "default": 4, "minimum": 1, "maximum": 10},
        },
        "required": ["query"],
    },
    function=retrieve_from_index,
)

<a name="tool-search"></a>
### 3.2 `web_search` — Perplexity Search API

`PerplexityWebSearch` returns ranked, cleaned web results as Haystack `Document`s and the raw URL list. No `LinkContentFetcher`, no HTML cleaning, no SERP API key.


In [ ]:
from haystack_integrations.components.websearch.perplexity import PerplexityWebSearch

_web_search = PerplexityWebSearch(top_k=5)


def web_search(query: str, top_k: int = 5) -> dict:
    """Run a Perplexity web search and return ranked, cited results."""
    result = _web_search.run(query=query, search_params={"max_results": top_k})
    return {
        "results": [
            {
                "title": d.meta.get("title", ""),
                "url": d.meta.get("url") or d.meta.get("link", ""),
                "snippet": d.content[:600],
            }
            for d in result["documents"]
        ],
        "links": result["links"],
    }


web_search_tool = Tool(
    name="web_search",
    description=(
        "Search the live web with the Perplexity Search API. "
        "Use this when retrieve_from_index returns nothing relevant, or when the question "
        "requires post-cutoff or current information."
    ),
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "top_k": {"type": "integer", "default": 5, "minimum": 1, "maximum": 20},
        },
        "required": ["query"],
    },
    function=web_search,
)

<a name="tool-ingest"></a>
### 3.3 `ingest_url` — close the loop

The agent picks a URL it considers high-signal (often returned by `web_search`), we fetch it with `trafilatura`, chunk it, embed it with `PerplexityDocumentEmbedder`, and write it back into Qdrant. The *next* call to `retrieve_from_index` can then find it.


In [ ]:
import trafilatura
from haystack.components.preprocessors import DocumentSplitter

_doc_embedder = PerplexityDocumentEmbedder(model="pplx-embed-v1-0.6b", progress_bar=False)
_splitter = DocumentSplitter(split_by="word", split_length=200, split_overlap=20)
_writer = DocumentWriter(document_store=document_store, policy=DuplicatePolicy.OVERWRITE)


def ingest_url(url: str, title: str | None = None) -> dict:
    """Fetch a URL, embed its contents, and add it to the Qdrant index."""
    raw = trafilatura.fetch_url(url)
    text = trafilatura.extract(raw) if raw else None
    if not text:
        return {"ok": False, "reason": "could not extract content", "url": url}

    doc = Document(content=text, meta={"source": url, "title": title or url})
    chunks = _splitter.run(documents=[doc])["documents"]
    embedded = _doc_embedder.run(documents=chunks)["documents"]
    _writer.run(documents=embedded)

    return {
        "ok": True,
        "url": url,
        "chunks_indexed": len(embedded),
        "total_docs_in_index": document_store.count_documents(),
    }


ingest_tool = Tool(
    name="ingest_url",
    description=(
        "Fetch the content at `url`, embed it with Perplexity embeddings, and add it to the "
        "local Qdrant index for reuse by future retrieve_from_index calls. "
        "Call this after web_search when you find an authoritative source worth keeping."
    ),
    parameters={
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "Absolute URL to fetch."},
            "title": {"type": "string", "description": "Optional human-readable title."},
        },
        "required": ["url"],
    },
    function=ingest_url,
)

<a name="agent"></a>
## 4. Build the agent (Perplexity Agent API)

`PerplexityChatGenerator` subclasses Haystack's `OpenAIResponsesChatGenerator`, which is exactly what the [`Agent`](https://docs.haystack.deepset.ai/docs/agent) component expects. We pass our three tools and a system prompt that spells out the retrieve-first / search-then-ingest policy.


In [ ]:
from haystack.components.agents import Agent
from haystack_integrations.components.generators.perplexity import PerplexityChatGenerator

SYSTEM_PROMPT = (
    "You are a research agent. For every user question:\n"
    "1. Call `retrieve_from_index` first to check the local knowledge base.\n"
    "2. If the retrieved snippets do not fully answer the question, call `web_search`.\n"
    "3. If `web_search` surfaces a high-quality URL the user is likely to ask about again, "
    "call `ingest_url` to add it to the index before answering.\n"
    "4. Cite every fact with the source URL it came from. "
    "Prefer concise answers with inline markdown links."
)

chat_generator = PerplexityChatGenerator(model="openai/gpt-5.4")

agent = Agent(
    chat_generator=chat_generator,
    tools=[retrieve_tool, web_search_tool, ingest_tool],
    system_prompt=SYSTEM_PROMPT,
    exit_conditions=["text"],
    max_agent_steps=8,
)
agent.warm_up()

<a name="run-q1"></a>
## 5. Run it

### 5.1 Question 1 — answerable from the seed index

This should be answered with a single `retrieve_from_index` call.


In [ ]:
from haystack.dataclasses import ChatMessage

q1 = "What does the perplexity-haystack package provide?"
result = agent.run(messages=[ChatMessage.from_user(q1)])
print(result["messages"][-1].text)

<a name="run-q2"></a>
### 5.2 Question 2 — requires live web search + ingestion

The agent should fail to answer from the index, fall back to `web_search`, ingest the best URL, and then answer with a citation.


In [ ]:
q2 = "Summarise the latest stable release notes for the perplexity-haystack package."
result = agent.run(messages=[ChatMessage.from_user(q2)])
print(result["messages"][-1].text)
print(f"\nDocuments in index after Q2: {document_store.count_documents()}")

<a name="run-q3"></a>
### 5.3 Question 3 — confirms the index actually grew

A follow-up that should now hit the ingested page via `retrieve_from_index` without another web call.


In [ ]:
q3 = "In the perplexity-haystack release notes you just ingested, which components were added or changed?"
result = agent.run(messages=[ChatMessage.from_user(q3)])
print(result["messages"][-1].text)

<a name="inspect"></a>
## 6. Inspect the trace

Each `ChatMessage` in `result['messages']` records the tool calls and their outputs. This is the per-turn evidence that the agent used the right tool for each question.


In [ ]:
for i, msg in enumerate(result["messages"]):
    tool_calls = getattr(msg, "tool_calls", []) or []
    tool_results = getattr(msg, "tool_call_results", []) or []
    summary = []
    if tool_calls:
        summary.append("calls=" + ", ".join(tc.tool_name for tc in tool_calls))
    if tool_results:
        summary.append(f"results={len(tool_results)}")
    if msg.text:
        summary.append(f"text[{len(msg.text)} chars]")
    print(f"[{i}] role={msg.role.value} " + " | ".join(summary))

<a name="wrap-up"></a>
## 7. Wrap up

You now have a Haystack agent that:

- **Decides** when to retrieve, search, or ingest — using Perplexity for all three.
- **Learns** across turns: the second question added a page to the Qdrant index, the third question retrieved it without another web call.
- **Cites** every web-derived fact — Perplexity's Search API returns the source URLs alongside the content, so the agent has nothing to hallucinate.

### Where to go next

- Swap `pplx-embed-v1-0.6b` for `pplx-embed-v1-4b` if you need higher-quality embeddings.
- Add `search_recency_filter` or `search_domain_filter` to `PerplexityWebSearch.search_params` to constrain results.
- Replace the in-memory Qdrant with a hosted Qdrant Cloud instance to persist what the agent learns.
- Read the [Perplexity x Haystack integration guide](https://docs.perplexity.ai/docs/getting-started/integrations/haystack) for advanced patterns (streaming, structured outputs, tool-calling with the Agent API).
